# L13b: Reinforcement Learning: Bandit Problems
In this lectyre, we'll introduce the general topic of reinforcement learning (RL) and focus on the simpler sub-problem of bandit problems. 

> __Learning Objectives:__
> 
> By the end of this lecture, you should be able to:
> Three actional learning objectives for this lecture go here.

Let's get started!
___

## Examples
Today, we will use the following examples to illustrate key concepts:

> [▶ Analyze a call credit vertical spread on `AMD`](CHEME-5660-L12b-Example-Delta-VerticalCallSpread-Fall-2025.ipynb). In this example, we'll compute the Delta for a vertical spread composite options contract on `AMD` using the CRR binomial lattice model and demonstrate how Delta can be used to construct a delta-neutral hedging position.

> [▶ Let's analyze a covered call position on `AMD`](CHEME-5660-L12a-Example-AnalysisCoveredCall-Fall-2025.ipynb). In this example, we analyze a covered call position on `AMD` using recent options chain data. We compute the potential profit/loss at expiration of the position as a function of various market factors, such as stock price at expiration, strike price, and option premium. We also visualize the profit/loss profile of the covered call position to understand its risk/reward characteristics.

> [▶ Let's analyze a synthetic covered call position on `INTC`](CHEME-5660-L12b-Example-AnalysisSyntheticCoveredCall-Fall-2025.ipynb). In this example, we analyze a synthetic covered call position on `INTC` using recent options chain data. We compute the potential profit/loss at expiration of the position as a function of various market factors, such as stock price at expiration, strike price, and option premium. We also visualize the profit/loss profile of the synthetic covered call position to understand its risk/reward characteristics.

> [▶ Let's analyze a cash secured put position on `AMD`](CHEME-5660-L12a-Example-CashSecuredPut-Fall-2025.ipynb). In this example, we analyze a cash secured put position on `AMD` using recent options chain data. We compute the potential profit/loss at expiration of the position as a function of various market factors, such as stock price at expiration, strike price, and option premium. We also visualize the profit/loss profile of the cash secured put position to understand its risk/reward characteristics.
___

## Concept Rewiew: Hidden Markov Models (HMMs)
Fill me in later!
___

## Reinforcement Learning Problem
Suppose we have an agent that can be in a state $s \in \mathcal{S}$ and can take an action $a \in \mathcal{A}$. After taking action $a$ in state $s$, the agent receives a reward $r$. But how does the agent learn to choose the best possible action in each state, i.e., develop a __policy__ to maximize its cumulative reward over time?

<div>
    <center>
        <img src="figs/Fig-Schematic-RL.svg" width="580"/>
    </center>
</div>

In reinforcement learning, an agent interacts with an environment by observing its current state $s \in \mathcal{S}$, selecting an action $a \in \mathcal{A}$, and receiving a reward that influences its future decisions. We'll explore three different approaches to this problem:

* __Bandit algorithms__ operate in stateless environments. On each round, they explore different actions to estimate their rewards and adapt their action-selection strategy based on the outcomes.
* __Multiplicative weights__ approaches the probability of selecting an action based on past performance, but they do so in a principled way that guarantees the algorithm performs nearly as well as the best fixed action in hindsight—even in changing environments, i.e., it minimizes regret.
* __Q-learning__ is a value-based method that estimates the long-term value (utility, satisfaction, happiness, etc) of each state-action pair, enabling the agent to learn optimal behavior in environments with temporal and sequential dynamics.

These approaches highlight different strategies for learning from interaction, but they all must balance a fundamental challenge in reinforcement learning: the tradeoff between exploring new actions to gather information and exploiting known actions to maximize reward.

___

## Exploration vs. Exploitation
In reinforcement learning, the problem on the surface is deceptively simple: an agent is in a state $s$ and can take an action $a\in A_{s}$, where $A_{s}$ is the set of actions currently available to the agent. The agent chooses an action $a$, implements it, and receives a reward $r$ and transitions to a new state $s^{\prime}$. The goal is to learn a policy that maximizes the cumulative reward over time, i.e., the best possible action in each state.

The problem is more complex than it appears. The agent must make decisions based on incomplete information and balance two competing objectives: exploration and exploitation. The exploration vs. exploitation trade-off is a fundamental challenge that agents must navigate:
1. **Exploration**: Trying new actions to discover potentially better rewards. This is essential for learning about the environment and finding optimal policies. Taking random actions or actions that have not been tried often to gather information about their outcomes and rewards is an example of exploration.
2. **Exploitation**: Leveraging current knowledge to maximize immediate payoff. This involves choosing actions that have previously yielded high rewards based on the agent's experience. If the agent only exploits, it may miss out on discovering better actions that could yield higher rewards in the long run.

Striking the right balance between exploration and exploitation is crucial for learning an optimal policy that performs well both now and in the long run. If an agent explores too much, it may miss out on immediate rewards; if it exploits too much, it may fail to discover better long-term strategies.

The exploration-exploitation trade-off is often formalized in algorithms that guide the agent's decision-making process. These algorithms provide principled strategies for managing the trade-off, ensuring that the agent can learn effectively while maximizing its cumulative reward over time.
___

<div>
    <center>
        <img src="figs/Fig-Schematic-RL.svg" width="580"/>
    </center>
</div>

## Binary Bernoulli Bandit Problem
The binary Bernoulli bandit problem is a special case of the stochastic bandit problem where the reward for taking action $a\in\mathcal{A}$ is binary $r_{t} = \left\{0,1\right\}$. The probability of getting reward `1` is unknown and needs to be estimated. The goal is to maximize the expected reward by selecting the best action at each time step.

* __Difference__: Unlike a completely general stochastic bandit problem, the binary Bernoulli bandit problem assumes the _agent models how the world responds_ using a (deceptively) simple reward distribution, [the Bernoulli distribution](https://en.wikipedia.org/wiki/Bernoulli_distribution). Thus, _the agent has a model of the world_ (which is so super cool!).
* __Binary__: The reward distribution is binary. However, this is not as limiting as it may first appear. The experiment represented by the action $a$ can be a complex statement or function that _evaluates_ to a boolean value. Thus, we can model many complex scenarios that evaluate to `true` or `false`.

The Bernoulli distribution is a discrete probability distribution that returns a value of `1` with probability $p$ and value `0` with probability $1-p$. The probability mass function of the Bernoulli distribution is given by:
$$
\begin{equation*}
\texttt{Bern}(r; p) = \begin{cases}
p & \text{if } r = 1,\\
1-p & \text{if } r = 0.
\end{cases}
\end{equation*}
$$
where $r\in\left\{0,1\right\}$ is the reward and $p\in[0,1]$ is the probability of getting reward `r = 1`. The expected reward of $X\sim\texttt{Bern}(r;p)$ is given by: $\mathbb{E}[X] = p$ and the variance is given by: $\text{Var}[X] = p(1-p)$. 
* _Ready to get your mind blown_? Ok, so here is the _cool part_: the agent models the parameter $p$ using a _probability distribution_ (e.g., [a Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution)) and updates this distribution as it observes rewards. This is the essence of the [Bayesian approach to bandit problems](https://onlinelibrary.wiley.com/doi/10.1002/asmb.874).

Yeah. That's cool. But how do we solve this problem?

### $\epsilon$-Greedy Binary Bernoulli Bandit
The $\epsilon$-greedy algorithm is simple and effective for solving the binary Bernoulli bandit problem. The algorithm selects the _best action_ with probability $1-\epsilon$ and selects a random action with probability $\epsilon$. The pseudocode for the $\epsilon$-greedy algorithm is given below.

#### Pseudo-code
The agent has $K$ arms (choices), $\mathcal{A} = \left\{1,2,\dots,K\right\}$, and the total number of rounds is $T\gg{K}$. Initialize the parameters of [the Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution) for each arm $a\in\mathcal{A}$ to $\alpha_{a} = 1$ and $\beta_{a} = 1$. The agent uses the following algorithm to choose which arm to pull (which action to take) during each round:

For $t = 1,2,\dots,T$:
1. _Initialize_: Roll a random number $p\in\left[0,1\right]$ and compute a threshold $\epsilon_{t}={t^{-1/3}}\cdot\left(K\cdot\log(t)\right)^{1/3}$.
2. _Exploration_: If $p\leq\epsilon_{t}$, choose a random (uniform) arm $a_{t}\in\mathcal{A}$. Execute the action $a_{t}$ and receive a reward $r_{t} = \left\{0,1\right\}$ from the _adversary_ (nature).
3. _Exploitation_: Else if $p>\epsilon_{t}$, choose action $a^{\star}_{t}$, the action with the _highest expected probability of success_ (greedy choice), using the agent's model of the world. The highest probability action is: $a^{\star} = \arg\max_{a\in\mathcal{A}}\left\{\frac{\alpha(a) + \mathbf{S}(a)}{\alpha(a) + \beta(a) + \mathbf{S}(a) + \mathbf{F}(a)}\right\}$ where $\mathbf{S}(a)$ and $\mathbf{F}(a)$ are the number of successes and failures for arm $a$. Execute the action $a^{\star}_{t}$ and receive a reward $r^{\star}_{t}\in\left\{0,1\right\}$ from the _adversary_ (nature).
4. Update the success $\mathbf{S}(a^{\star})$ and failure $\mathbf{F}(a^{\star})$ arrays for the chosen arm $a^{\star}_{t}$ using the reward $r^{\star}_{t}$:
$$
\begin{equation*}
S(a^{\star}_{t}) \gets S(a^{\star}_{t}) + r^{\star}_{t},\quad F(a^{\star}_{t}) \gets F(a^{\star}_{t}) + (1-r^{\star}_{t})
\end{equation*}
$$

Using a model of the world allows the agent to make decisions about which actions to take. This is the essence of the Bayesian approach to bandit problems. The agent has a model of the likely reward distribution for _each_ action and uses this model to select the best action at each time step.

### Thompson Sampling Binary Bernoulli Bandit
An alternative Bayesian approach is Thompson sampling, which samples from the posterior distribution to make decisions.

> __Thompson Sampling__: The Thompson sampling algorithm selects the _best action_ by sampling from the posterior distribution for each arm and choosing the one with the highest sample. There is no explicit exploration step; instead, exploration occurs naturally through the sampling process. The pseudocode for the Thompson sampling algorithm is given below.

#### Pseudo-code
The agent has $K$ arms (choices), $\mathcal{A} = \left\{1,2,\dots,K\right\}$, and the total number of rounds is $T\gg{K}$. Initialize the parameters of [the Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution) for each arm $a\in\mathcal{A}$ to $\alpha_{a} = 1$ and $\beta_{a} = 1$. The agent uses the following algorithm to choose which arm to pull (which action to take) during each round:

For $t = 1,2,\dots,T$:
1. Sample from the posterior for each arm: $\mathbf{p}\gets\left\{\text{Beta}(\alpha(a)+\mathbf{S}(a),\beta(a)+\mathbf{F}(a))\mid\forall{a}\in\mathcal{A}\right\}$ where $\mathbf{S}(a)$ and $\mathbf{F}(a)$ are the number of successes and failures for arm $a$.
2. Choose the action with the highest sampled probability: $a^{\star} = \arg\max_{a\in\mathcal{A}}\left\{\mathbf{p}(a)\right\}$.
3. Execute the action $a^{\star}_{t}$ and receive a reward $r^{\star}_{t}\in\left\{0,1\right\}$ from the _adversary_ (nature).
4. Update the success $\mathbf{S}(a^{\star})$ and failure $\mathbf{F}(a^{\star})$ arrays for the chosen arm $a^{\star}_{t}$ using the reward $r^{\star}_{t}$:
$$
\begin{equation*}
S(a^{\star}_{t}) \gets S(a^{\star}_{t}) + r^{\star}_{t},\quad F(a^{\star}_{t}) \gets F(a^{\star}_{t}) + (1-r^{\star}_{t})
\end{equation*}
$$

Thompson sampling is fully Bayesian and often performs better than ε-greedy in practice. 

> Let's think about a few things here:
> 
> * __Context:__ If we step back, some decisions depend upon context. For example, understanding where we are on the planet would be handy if we were predicting the weather. Predicting product demand might depend upon the season, or determining which drugs to prescribe would depend upon the indication. Thus, _context_ is essential.
> * __Combinatorial Actions:__ In many scenarios, we need to select multiple actions simultaneously. For example, in drug discovery, we might want to test combinations of compounds. In marketing, we might want to select a set of advertisements to display. This leads us to the concept of combinatorial bandits, where the agent selects a subset of actions at each time step.

How do we handle these more complex scenarios? Simple! We extend our models to incorporate context and combinatorial actions (map an integer to a subset of actions). While a little more complex, the core principles remain the same: balancing exploration and exploitation to maximize cumulative reward.

Let's look at examples of a Binary Bernoulli Bandit problem using both the ε-greedy and Thompson sampling algorithms.

___

## Summary
One concise, direct summary sentence goes here.

> __Key Takeaways:__
> Three key takeaways for this lecture go here.

One concise, direct concluding sentence goes here.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___